In [ ]:
import pandas as pd
import numpy as np
import gzip
import json
import pickle

print("Loading metadata... this may take 1-2 minutes...")

def load_metadata(path, nrows=None):
    data = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if nrows and i >= nrows:
                break
            try:
                data.append(json.loads(line))
            except:
                continue
    return pd.DataFrame(data)

meta_df = load_metadata(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\meta_Musical_Instruments.json.gz')

print(f"✅ Loaded metadata")
print(f"Shape       : {meta_df.shape}")
print(f"Columns     : {meta_df.columns.tolist()}")
print(meta_df.head(2))

Loading metadata... this may take 1-2 minutes...
✅ Loaded metadata
Shape       : (786445, 19)
Columns     : ['category', 'tech1', 'description', 'fit', 'title', 'also_buy', 'tech2', 'brand', 'feature', 'rank', 'also_view', 'main_cat', 'similar_item', 'date', 'price', 'asin', 'imageURL', 'imageURLHighRes', 'details']
                                            category tech1  \
0  [Electronics, Camera &amp; Photo, Video Survei...         
1                  [Electronics, Camera &amp; Photo]         

                                         description fit  \
0  [The following camera brands and models have b...       
1  [This second edition of the Handbook of Astron...       

                                               title      also_buy tech2  \
0  Genuine Geovision 1 Channel 3rd Party NVR IP S...            []         
1  Books "Handbook of Astronomical Image Processi...  [0999470906]         

          brand                                            feature  \
0     GeoVision

In [11]:
# Let's see exactly what useful data exists
print("Column by column check:")
for col in meta_df.columns:
    non_null = meta_df[col].notna().sum()
    pct      = round(non_null / len(meta_df) * 100, 1)
    print(f"  {col:30} → {non_null:,} non-null ({pct}%)")

Column by column check:
  category                       → 786,445 non-null (100.0%)
  tech1                          → 786,445 non-null (100.0%)
  description                    → 786,445 non-null (100.0%)
  fit                            → 786,445 non-null (100.0%)
  title                          → 786,445 non-null (100.0%)
  also_buy                       → 786,445 non-null (100.0%)
  tech2                          → 786,445 non-null (100.0%)
  brand                          → 786,445 non-null (100.0%)
  feature                        → 786,445 non-null (100.0%)
  rank                           → 786,445 non-null (100.0%)
  also_view                      → 786,445 non-null (100.0%)
  main_cat                       → 786,445 non-null (100.0%)
  similar_item                   → 786,445 non-null (100.0%)
  date                           → 786,445 non-null (100.0%)
  price                          → 786,445 non-null (100.0%)
  asin                           → 786,445 non-null (100.0%)


In [12]:
# Keep only columns that are useful for display
# We'll handle missing columns gracefully

def safe_get_col(df, col):
    return df[col] if col in df.columns else pd.Series([''] * len(df))

meta_clean = pd.DataFrame()
meta_clean['product_id']  = safe_get_col(meta_df, 'asin')
meta_clean['title']       = safe_get_col(meta_df, 'title')
meta_clean['brand']       = safe_get_col(meta_df, 'brand')
meta_clean['price']       = safe_get_col(meta_df, 'price')
meta_clean['description'] = safe_get_col(meta_df, 'description')
meta_clean['rank']        = safe_get_col(meta_df, 'rank')

# Image — try high res first then fall back to regular
if 'imageURLHighRes' in meta_df.columns:
    meta_clean['image_url'] = meta_df['imageURLHighRes'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ''
    )
elif 'imageURL' in meta_df.columns:
    meta_clean['image_url'] = meta_df['imageURL'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ''
    )
else:
    meta_clean['image_url'] = ''

# Feature bullets
if 'feature' in meta_df.columns:
    meta_clean['features'] = meta_df['feature'].apply(
        lambda x: ' | '.join(x[:3]) if isinstance(x, list) else ''
    )
else:
    meta_clean['features'] = ''

# Drop rows with no product_id
meta_clean = meta_clean[meta_clean['product_id'] != '']
meta_clean = meta_clean.drop_duplicates(subset='product_id')

print(f"Clean metadata shape : {meta_clean.shape}")
print(meta_clean.head(3).to_string())

Clean metadata shape : (756077, 8)
   product_id                                                                                                              title                                         brand   price                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [13]:
# Load your clean ratings to get the product IDs you actually use
df = pd.read_csv(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\clean_ratings.csv')
your_products = df['product_id'].unique()

print(f"Products in your dataset : {len(your_products):,}")
print(f"Products in metadata     : {len(meta_clean):,}")

# Keep only metadata for products in your dataset
matched_meta = meta_clean[meta_clean['product_id'].isin(your_products)]
print(f"Matched products         : {len(matched_meta):,}")

# How many of your products have metadata?
coverage = round(len(matched_meta) / len(your_products) * 100, 1)
print(f"Coverage                 : {coverage}%")

Products in your dataset : 2,717
Products in metadata     : 756,077
Matched products         : 0
Coverage                 : 0.0%


In [14]:
def clean_title(title):
    if not isinstance(title, str) or title.strip() == '':
        return 'Unknown Product'
    # Truncate very long titles
    return title[:80] + '...' if len(title) > 80 else title

def clean_price(price):
    if not isinstance(price, str) or price.strip() == '':
        return 'N/A'
    # Keep only first price if range given
    price = price.split('-')[0].strip()
    return price

def clean_description(desc):
    if isinstance(desc, list):
        desc = ' '.join(desc)
    if not isinstance(desc, str) or desc.strip() == '':
        return 'No description available'
    return desc[:200] + '...' if len(desc) > 200 else desc

matched_meta['title']       = matched_meta['title'].apply(clean_title)
matched_meta['price']       = matched_meta['price'].apply(clean_price)
matched_meta['description'] = matched_meta['description'].apply(clean_description)

# Fill remaining nulls
matched_meta = matched_meta.fillna('')

print("✅ Cleaned metadata sample:")
print(matched_meta[['product_id', 'title', 'price', 'brand']].head(10).to_string())

✅ Cleaned metadata sample:
Empty DataFrame
Columns: [product_id, title, price, brand]
Index: []


In [15]:
import os, pickle

matched_meta.to_csv('../data/product_metadata.csv', index=False)

meta_lookup = matched_meta.set_index('product_id').to_dict(orient='index')

with open('../models/meta_lookup.pkl', 'wb') as f:
    pickle.dump(meta_lookup, f)

print(f"✅ Saved product_metadata.csv")
print(f"✅ Saved meta_lookup.pkl")
print(f"Total products in lookup : {len(meta_lookup)}")

# Test the lookup
if len(meta_lookup) > 0:
    sample_id = list(meta_lookup.keys())[0]
    sample    = meta_lookup[sample_id]
    print(f"\nSample product:")
    print(f"  Product ID  : {sample_id}")
    print(f"  Title       : {sample['title']}")
    print(f"  Brand       : {sample['brand']}")
    print(f"  Price       : {sample['price']}")
    print(f"  Image URL   : {'✅ exists' if sample['image_url'] else '❌ missing'}")
    print(f"  Features    : {sample['features'][:80]}...")
else:
    print("❌ Still empty — paste output for diagnosis")

✅ Saved product_metadata.csv
✅ Saved meta_lookup.pkl
Total products in lookup : 0
❌ Still empty — paste output for diagnosis


In [19]:
import pandas as pd
import os

# Reload clean ratings properly
df = pd.read_csv(os.path.join(data_path, 'clean_ratings.csv'))

print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Keep only essential columns needed by recommender
df_light = df[['user_id', 'product_id', 'rating', 'timestamp']].copy()

# Save lighter version
light_path = os.path.join(data_path, 'clean_ratings_light.csv')
df_light.to_csv(light_path, index=False)

print(f"\n✅ Saved light version")
print(f"Original columns : {df.columns.tolist()}")
print(f"Light columns    : {df_light.columns.tolist()}")
print(f"Light size       : {os.path.getsize(light_path)/1024**2:.1f} MB")

Loaded: (72696, 6)
Columns: ['user_id', 'product_id', 'rating', 'review_text', 'summary', 'timestamp']
Memory: 41.9 MB

✅ Saved light version
Original columns : ['user_id', 'product_id', 'rating', 'review_text', 'summary', 'timestamp']
Light columns    : ['user_id', 'product_id', 'rating', 'timestamp']
Light size       : 2.9 MB


In [7]:
import sys
import os
sys.path.append('..')
from src.recommender import HybridRecommender

# Fix: pass absolute paths so it works from any directory
base_dir    = os.path.abspath('..')
models_path = os.path.join(base_dir, 'models')
data_path   = os.path.join(base_dir, 'data')

print(f"Base dir    : {base_dir}")
print(f"Models path : {models_path}")
print(f"Data path   : {data_path}")

rec = HybridRecommender(models_path=models_path, data_path=data_path)

def get_enriched_recommendations(user_id, n=10):
    recs = rec.recommend(user_id, n=n)

    enriched = []
    for r in recs:
        pid  = r['product_id']
        meta = meta_lookup.get(pid, {})
        enriched.append({
            'product_id'  : pid,
            'score'       : r['score'],
            'title'       : meta.get('title', 'Unknown Product'),
            'brand'       : meta.get('brand', 'Unknown Brand'),
            'price'       : meta.get('price', 'N/A'),
            'image_url'   : meta.get('image_url', ''),
            'description' : meta.get('description', ''),
            'features'    : meta.get('features', ''),
            'category'    : meta.get('category', '')
        })
    return enriched

# Test it
df         = pd.read_csv(os.path.join(data_path, 'clean_ratings.csv'))
sample_user = df['user_id'].iloc[0]
print(f"\nEnriched recommendations for: {sample_user}\n")
results = get_enriched_recommendations(sample_user, n=5)

for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']}")
    print(f"   Brand    : {r['brand']}")
    print(f"   Price    : {r['price']}")
    print(f"   Category : {r['category']}")
    print(f"   Score    : {r['score']}")
    print(f"   Image    : {'✅' if r['image_url'] else '❌'}")
    print()

Base dir    : d:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy
Models path : d:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models
Data path   : d:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data
Loaded ratings: (72696, 4)
Matrix shape: (10327, 2717)
Training ALS...


  0%|          | 0/30 [00:00<?, ?it/s]

ALS done — user_factors: (10327, 50)

Enriched recommendations for: A3FO5AKVTFRCRJ

1. D'Addario EJ46 Pro-Arte Nylon Classical Guitar Strings, Hard Tension
   Brand    : D'Addario
   Price    : $8.99
   Category : Musical Instruments
   Score    : 0.8
   Image    : ✅

2. D'Addario EXL160TP Medium Gauge Nickel Wound Bass Strings XL 50-105 Long-Scale, ...
   Brand    : D'Addario
   Price    : $37.99
   Category : Musical Instruments
   Score    : 0.68
   Image    : ✅

3. D'Addario EXL170 Nickel Wound Bass Guitar Strings, Light, 45-100, Long Scale
   Brand    : D'Addario
   Price    : $19.99
   Category : Musical Instruments
   Score    : 0.68
   Image    : ✅

4. Unknown Product
   Brand    : Unknown Brand
   Price    : N/A
   Category : 
   Score    : 0.5
   Image    : ❌

5. D'Addario EJ26 Phosphor Bronze Acoustic Guitar Strings, Custom Light, 11-52
   Brand    : D'Addario
   Price    : $6.99
   Category : Musical Instruments
   Score    : 0.5
   Image    : ✅



In [21]:
import os

# Write directly using open with explicit path
recommender_path = os.path.join(os.path.abspath('..'), 'src', 'recommender.py')
print(f"Writing to: {recommender_path}")

code = '''import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

class HybridRecommender:
    def __init__(self, models_path="../models", data_path="../data"):
        light_path = f"{data_path}/clean_ratings_light.csv"
        self.df = pd.read_csv(light_path)
        print(f"Loaded ratings: {self.df.shape}")

        with open(f"{models_path}/combined_matrix.pkl", "rb") as f:
            self.combined_matrix = pickle.load(f)
        with open(f"{models_path}/product_profiles.pkl", "rb") as f:
            self.product_profiles = pickle.load(f)
        with open(f"{models_path}/product_idx_map.pkl", "rb") as f:
            maps = pickle.load(f)
            self.product_idx_map = maps["product_idx_map"]
            self.product_id_map  = maps["product_id_map"]

        self.df["confidence"] = self.df["rating"].apply(lambda x: 1 + 40 * x)
        train_users    = self.df["user_id"].unique()
        train_products = self.df["product_id"].unique()

        self.train_user_encoder    = {uid: idx for idx, uid in enumerate(train_users)}
        self.train_user_decoder    = {idx: uid for uid, idx in self.train_user_encoder.items()}
        self.train_product_encoder = {pid: idx for idx, pid in enumerate(train_products)}
        self.train_product_decoder = {idx: pid for pid, idx in self.train_product_encoder.items()}

        user_idx    = self.df["user_id"].map(self.train_user_encoder)
        product_idx = self.df["product_id"].map(self.train_product_encoder)
        valid       = user_idx.notna() & product_idx.notna()
        df_clean    = self.df[valid].copy()
        user_idx    = user_idx[valid].astype(int)
        product_idx = product_idx[valid].astype(int)

        self.train_matrix = csr_matrix(
            (df_clean["confidence"], (user_idx, product_idx)),
            shape=(len(self.train_user_encoder), len(self.train_product_encoder))
        )
        print(f"Matrix shape: {self.train_matrix.shape}")

        from implicit import als
        print("Training ALS...")
        self.train_als = als.AlternatingLeastSquares(
            factors=50, regularization=0.1,
            iterations=30, use_gpu=False
        )
        self.train_als.fit(self.train_matrix)
        print(f"ALS done — user_factors: {self.train_als.user_factors.shape}")

    def recommend(self, user_id, n=10):
        count = len(self.df[self.df["user_id"] == user_id])
        if count < 5:
            als_w, cb_w = 0.2, 0.8
        elif count < 20:
            als_w, cb_w = 0.5, 0.5
        else:
            als_w, cb_w = 0.8, 0.2

        als_r = self._als_recs(user_id, n=50)
        cb_r  = self._content_recs(user_id, n=50)

        als_scores = {p: (1 - i/len(als_r)) for i, p in enumerate(als_r)} if als_r else {}
        cb_scores  = {p: (1 - i/len(cb_r))  for i, p in enumerate(cb_r)}  if cb_r  else {}

        all_products = set(als_scores.keys()) | set(cb_scores.keys())
        if not all_products:
            return []

        hybrid_scores = {
            pid: als_w * als_scores.get(pid, 0) + cb_w * cb_scores.get(pid, 0)
            for pid in all_products
        }
        top = sorted(hybrid_scores, key=hybrid_scores.get, reverse=True)[:n]
        return [{"product_id": pid, "score": round(hybrid_scores[pid], 4)} for pid in top]

    def _als_recs(self, user_id, n=20):
        if user_id not in self.train_user_encoder:
            return []
        uid = self.train_user_encoder[user_id]
        if uid >= self.train_als.user_factors.shape[0]:
            return []
        user_items = csr_matrix(self.train_matrix[uid])
        ids, scores = self.train_als.recommend(
            uid, user_items, N=n,
            filter_already_liked_items=True
        )
        results = []
        for p in ids:
            p = int(p)
            if p in self.train_product_decoder:
                results.append(self.train_product_decoder[p])
        return results

    def _content_recs(self, user_id, n=20):
        liked = self.df[
            (self.df["user_id"] == user_id) &
            (self.df["rating"] >= 4)
        ]
        if liked.empty:
            return []
        acc = np.zeros(len(self.product_profiles))
        for _, row in liked.iterrows():
            if row["product_id"] not in self.product_idx_map:
                continue
            idx  = self.product_idx_map[row["product_id"]]
            sims = cosine_similarity(
                self.combined_matrix[idx],
                self.combined_matrix
            ).flatten()
            acc += sims * row["rating"]
        rated = set(self.df[self.df["user_id"] == user_id]["product_id"])
        for pid in rated:
            if pid in self.product_idx_map:
                acc[self.product_idx_map[pid]] = 0
        top = acc.argsort()[::-1][:n]
        return [self.product_id_map[i] for i in top if acc[i] > 0]
'''

with open(recommender_path, 'w') as f:
    f.write(code)

# Verify file was written correctly
with open(recommender_path, 'r') as f:
    content = f.read()

print(f"✅ File written successfully")
print(f"   File size : {len(content)} characters")
print(f"   Has _als_recs: {'_als_recs' in content}")
print(f"   Has train_matrix: {'train_matrix' in content}")
print(f"   Has csr_matrix(self.train_matrix: {'csr_matrix(self.train_matrix' in content}")

Writing to: d:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\src\recommender.py
✅ File written successfully
   File size : 4908 characters
   Has _als_recs: True
   Has train_matrix: True
   Has csr_matrix(self.train_matrix: True


In [1]:
# Cell 1 — after kernel restart
import os, sys, pickle
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

base_dir    = os.path.abspath('..')
models_path = os.path.join(base_dir, 'models')
data_path   = os.path.join(base_dir, 'data')

# Load metadata lookup
with open(os.path.join(models_path, 'meta_lookup.pkl'), 'rb') as f:
    meta_lookup = pickle.load(f)

print(f"Meta lookup size: {len(meta_lookup)}")

Meta lookup size: 0


In [2]:
# Cell 2 — fresh import after restart
from src.recommender import HybridRecommender

print("Initializing recommender...")
rec = HybridRecommender(models_path=models_path, data_path=data_path)
print("✅ Ready!")

Initializing recommender...
Loaded ratings: (72696, 4)
Matrix shape: (10327, 2717)
Training ALS...


c:\Users\bvais\anaconda3\envs\ml\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

ALS done — user_factors: (10327, 50)
✅ Ready!


In [3]:
# Cell 3 — test enriched recommendations
df = pd.read_csv(os.path.join(data_path, 'clean_ratings_light.csv'))

def get_enriched_recommendations(user_id, n=10):
    recs = rec.recommend(user_id, n=n)
    enriched = []
    for r in recs:
        pid  = r['product_id']
        meta = meta_lookup.get(pid, {})
        enriched.append({
            'product_id'  : pid,
            'score'       : r['score'],
            'title'       : meta.get('title', 'Unknown Product'),
            'brand'       : meta.get('brand', 'Unknown Brand'),
            'price'       : meta.get('price', 'N/A'),
            'image_url'   : meta.get('image_url', ''),
            'description' : meta.get('description', ''),
            'features'    : meta.get('features', ''),
            'category'    : meta.get('category', '')
        })
    return enriched

sample_user = df['user_id'].iloc[0]
print(f"Recommendations for: {sample_user}\n")
results = get_enriched_recommendations(sample_user, n=5)

for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']}")
    print(f"   Brand : {r['brand']}")
    print(f"   Price : {r['price']}")
    print(f"   Score : {r['score']}")
    print(f"   Image : {'✅' if r['image_url'] else '❌'}")
    print()

Recommendations for: A3FO5AKVTFRCRJ

1. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.93
   Image : ❌

2. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.78
   Image : ❌

3. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.56
   Image : ❌

4. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.5
   Image : ❌

5. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.49
   Image : ❌



In [4]:
# Check if meta_lookup has any data at all
print(f"Meta lookup size: {len(meta_lookup)}")

if len(meta_lookup) > 0:
    # Show sample keys from meta_lookup
    sample_keys = list(meta_lookup.keys())[:5]
    print(f"\nSample meta_lookup keys: {sample_keys}")
    
    # Show sample product IDs from recommendations
    raw_recs = rec.recommend(sample_user, n=5)
    rec_ids  = [r['product_id'] for r in raw_recs]
    print(f"\nRecommended product IDs: {rec_ids}")
    
    # Check if any overlap
    overlap = set(rec_ids) & set(meta_lookup.keys())
    print(f"\nOverlap: {len(overlap)}")
    
    # Check a specific lookup
    test_id = rec_ids[0]
    print(f"\nLooking up: {test_id}")
    print(f"Found: {meta_lookup.get(test_id, 'NOT FOUND')}")

else:
    print("❌ meta_lookup is EMPTY — need to reload it")
    
    # Reload from CSV
    meta_df_check = pd.read_csv(os.path.join(data_path, 'product_metadata.csv'))
    print(f"\nproduct_metadata.csv shape: {meta_df_check.shape}")
    print(f"Sample rows:")
    print(meta_df_check[['product_id','title','brand','price']].head(5).to_string())

Meta lookup size: 0
❌ meta_lookup is EMPTY — need to reload it

product_metadata.csv shape: (0, 8)
Sample rows:
Empty DataFrame
Columns: [product_id, title, brand, price]
Index: []


In [5]:
import pandas as pd
import numpy as np
import gzip
import json
import pickle
import os

data_path   = os.path.join(os.path.abspath('..'), 'data')
models_path = os.path.join(os.path.abspath('..'), 'models')

# ── Step 1: Load your clean ratings ──────────────────────
df = pd.read_csv(os.path.join(data_path, 'clean_ratings_light.csv'))
your_products = set(df['product_id'].unique())
print(f"Your products      : {len(your_products):,}")

# ── Step 2: Load metadata file ────────────────────────────
meta_path = os.path.join(data_path, 'meta_Musical_Instruments.json.gz')
print(f"\nLoading metadata from: {meta_path}")
print("This may take 1-2 minutes...")

records = []
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            item = json.loads(line.strip())
            asin = item.get('asin', '').strip()
            if asin in your_products:
                records.append({
                    'product_id'  : asin,
                    'title'       : item.get('title', ''),
                    'brand'       : item.get('brand', ''),
                    'price'       : item.get('price', ''),
                    'description' : item.get('description', ''),
                    'image_url'   : (item.get('imageURLHighRes', [None]) or
                                     item.get('imageURL',        [None]))[0] or '',
                    'features'    : ' | '.join(
                                        item.get('feature', [])[:3]
                                    ) if item.get('feature') else '',
                    'category'    : item.get('category', [''])[0]
                                    if isinstance(item.get('category'), list)
                                    else item.get('category', ''),
                })
        except Exception as e:
            continue

print(f"Matched records    : {len(records):,}")

# ── Step 3: Build dataframe ───────────────────────────────
meta_df = pd.DataFrame(records)

if len(meta_df) == 0:
    print("\n❌ Still no matches — printing sample ASINs from metadata file")
    # Sample first 10 ASINs from metadata to compare
    with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 10:
                break
            try:
                item = json.loads(line.strip())
                print(f"  metadata asin: '{item.get('asin', '')}'")
            except:
                continue
    print("\nYour product IDs sample:")
    print(list(your_products)[:10])
else:
    # ── Step 4: Clean ─────────────────────────────────────
    def clean_title(t):
        if not isinstance(t, str) or t.strip() == '':
            return 'Unknown Product'
        return t[:80] + '...' if len(t) > 80 else t

    def clean_price(p):
        if not isinstance(p, str) or p.strip() == '':
            return 'N/A'
        return p.split('-')[0].strip()

    def clean_description(d):
        if isinstance(d, list):
            d = ' '.join(d)
        if not isinstance(d, str) or d.strip() == '':
            return 'No description available'
        return d[:200] + '...' if len(d) > 200 else d

    meta_df['title']       = meta_df['title'].apply(clean_title)
    meta_df['price']       = meta_df['price'].apply(clean_price)
    meta_df['description'] = meta_df['description'].apply(clean_description)
    meta_df                = meta_df.fillna('')
    meta_df                = meta_df.drop_duplicates(subset='product_id')

    coverage = round(len(meta_df) / len(your_products) * 100, 1)
    print(f"Coverage           : {coverage}%")
    print(f"\nSample matches:")
    print(meta_df[['product_id','title','brand','price']].head(10).to_string())

    # ── Step 5: Save ──────────────────────────────────────
    csv_path = os.path.join(data_path, 'product_metadata.csv')
    pkl_path = os.path.join(models_path, 'meta_lookup.pkl')

    meta_df.to_csv(csv_path, index=False)

    meta_lookup = meta_df.set_index('product_id').to_dict(orient='index')
    with open(pkl_path, 'wb') as f:
        pickle.dump(meta_lookup, f)

    print(f"\n✅ Saved product_metadata.csv → {len(meta_df)} products")
    print(f"✅ Saved meta_lookup.pkl      → {len(meta_lookup)} entries")

    # ── Step 6: Test lookup ───────────────────────────────
    print(f"\nTest lookup:")
    sample_id = list(meta_lookup.keys())[0]
    sample    = meta_lookup[sample_id]
    print(f"  product_id  : {sample_id}")
    print(f"  title       : {sample['title']}")
    print(f"  brand       : {sample['brand']}")
    print(f"  price       : {sample['price']}")
    print(f"  image_url   : {'✅ exists' if sample['image_url'] else '❌ missing'}")

Your products      : 2,717

Loading metadata from: d:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\meta_Musical_Instruments.json.gz
This may take 1-2 minutes...
Matched records    : 2,611
Coverage           : 70.7%

Sample matches:
   product_id                                                                                title                    brand    price
0  0739079891                        Alfreds Teach Yourself to Play Ukulele, Complete Starter Pack  Alfred Music Publishing      N/A
1  1480360295  ChordBuddy Guitar Learning System for Right Handed Guitars. Includes ChordBuddy,...               ChordBuddy    $6.04
2  9792372326                                    QSC K10 2-Way Powered Speaker - 1000 Watts, 1x10"                      QSC  $125.99
3  B00000J50W                   Music Maker - Hand Made Lap Harp - Easy to Play Musical Instrument              Music Maker   $39.99
4  B00004UE29                                                       Yamaha YRS24B Soprano Recorder

In [6]:
# Reload fresh meta_lookup
with open(os.path.join(models_path, 'meta_lookup.pkl'), 'rb') as f:
    meta_lookup = pickle.load(f)

print(f"Meta lookup loaded: {len(meta_lookup)} products")

# Test enriched recommendations
def get_enriched_recommendations(user_id, n=10):
    recs = rec.recommend(user_id, n=n)
    enriched = []
    for r in recs:
        pid  = r['product_id']
        meta = meta_lookup.get(pid, {})
        enriched.append({
            'product_id'  : pid,
            'score'       : r['score'],
            'title'       : meta.get('title', 'Unknown Product'),
            'brand'       : meta.get('brand', 'Unknown Brand'),
            'price'       : meta.get('price', 'N/A'),
            'image_url'   : meta.get('image_url', ''),
            'description' : meta.get('description', ''),
            'features'    : meta.get('features', ''),
            'category'    : meta.get('category', '')
        })
    return enriched

sample_user = df['user_id'].iloc[0]
print(f"\nEnriched recommendations for: {sample_user}\n")
results = get_enriched_recommendations(sample_user, n=5)

for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']}")
    print(f"   Brand : {r['brand']}")
    print(f"   Price : {r['price']}")
    print(f"   Score : {r['score']}")
    print(f"   Image : {'✅' if r['image_url'] else '❌'}")
    print()

Meta lookup loaded: 1921 products

Enriched recommendations for: A3FO5AKVTFRCRJ

1. D'Addario EJ46 Pro-Arte Nylon Classical Guitar Strings, Hard Tension
   Brand : D'Addario
   Price : $8.99
   Score : 0.93
   Image : ✅

2. D'Addario EXP15 with NY Steel Phosphor Bronze Acoustic Guitar Strings, Coated, E...
   Brand : D'Addario
   Price : $15.19
   Score : 0.78
   Image : ✅

3. D'Addario Assorted Pearl Celluloid Guitar Picks, 10 Pack, Medium
   Brand : D'Addario Accessories
   Price : $3.70
   Score : 0.56
   Image : ✅

4. D'Addario EJ26 Phosphor Bronze Acoustic Guitar Strings, Custom Light, 11-52
   Brand : D'Addario
   Price : $6.99
   Score : 0.5
   Image : ✅

5. Unknown Product
   Brand : Unknown Brand
   Price : N/A
   Score : 0.49
   Image : ❌

